# Alzheimer's OASIS-1 — DeiT 黑盒基线（二分类）

和 `alzheimer_baseline_densenet121_oasis_colab.ipynb` 完全一样的方法论：复用
`oof_predictions_ANFIS-on-GT.csv` 里记录的**同一批 226 个受试者、同一套 5 折切分**，
复用 DenseNet-121 notebook 已经生成好的轴状位切片，只换模型架构成 DeiT
（Data-efficient image Transformer）。

DeiT 不在 torchvision 里，用 `timm`（PyTorch Image Models，成熟维护的标准库，
和 SynthSeg 那次的外部依赖风险不是一回事）加载。选 **deit_small**（不是 base）：
每折训练集只有 ~180 人，模型越大过拟合风险越高，这个考虑和项目里选 Swin-T（tiny）
而不是更大变体是一致的。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                        'nibabel', 'openpyxl', 'pandas', 'scikit-learn', 'timm'])

In [ ]:
import hashlib
import os
from collections import defaultdict
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from IPython.display import Image as IPImage, display
from PIL import Image
from sklearn.metrics import confusion_matrix, roc_auc_score, classification_report
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
from tqdm.notebook import tqdm

DRIVE_ROOT = '/content/drive/MyDrive'
OASIS_DIR  = f'{DRIVE_ROOT}/OASIS_Freesurfer'

OUTPUT_DIR      = f'{DRIVE_ROOT}/alzheimer_anfis_outputs'
SLICES_DIR      = f'{OUTPUT_DIR}/baseline_densenet121_slices'
CHECKPOINT_DIR  = f'{OUTPUT_DIR}/checkpoints'
RESULTS_DIR     = f'{OUTPUT_DIR}/results'
ANFIS_GT_OOF_CSV = f'{RESULTS_DIR}/oof_predictions_ANFIS-on-GT.csv'

for d in [SLICES_DIR, CHECKPOINT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

CLASS_ORDER   = ['Nondemented', 'Demented']
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True

AXIAL_DIM, AXIAL_RANGE = 2, (0.27, 0.47)
N_SLICES = 24

BATCH_SIZE  = 32
NUM_WORKERS = 4
USE_AMP     = True

PHASE1_EPOCHS = 12
PHASE2_EPOCHS = 15
PATIENCE      = 6
MIN_DELTA     = 0.001
PHASE1_LR     = 1e-4
PHASE2_LR_CNN = 1e-6
PHASE2_LR_HEAD = 1e-5
WEIGHT_DECAY  = 1e-4

DEIT_MODEL_NAME = 'deit_small_patch16_224.fb_in1k'

print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_properties(0).name}')

In [ ]:
# ── 加载 ANFIS-on-GT 用过的同一批受试者 + 同一套 5 折切分（关键：公平对比的基础）──
assert os.path.exists(ANFIS_GT_OOF_CSV), (
    f'找不到 {ANFIS_GT_OOF_CSV}，先跑完 alzheimer_anfis_train_colab.ipynb 的 ANFIS-on-GT 配置。')

anfis_oof = pd.read_csv(ANFIS_GT_OOF_CSV)
subject_fold = anfis_oof.set_index('subject_id')[['label', 'oof_fold']].to_dict('index')
print(f'复用 ANFIS-on-GT 的 {len(subject_fold)} 个受试者、{anfis_oof["oof_fold"].nunique()} 折切分')
print(anfis_oof['label'].value_counts())

In [ ]:
# ── 找这批受试者的 brainmask.mgz（os.walk，不用 Path.glob）──
print('正在扫描 brainmask.mgz 文件...')
mgz_map = {}
for root, dirs, files in os.walk(OASIS_DIR):
    if 'brainmask.mgz' in files and os.path.basename(root) == 'mri':
        sid = os.path.basename(os.path.dirname(root))
        if sid in subject_fold:
            mgz_map[sid] = os.path.join(root, 'brainmask.mgz')

missing = set(subject_fold.keys()) - set(mgz_map.keys())
print(f'找到 {len(mgz_map)} / {len(subject_fold)} 个受试者的 brainmask.mgz')
if missing:
    print(f'缺失 {len(missing)} 个（会跳过）：', list(missing)[:10])

In [ ]:
# ── MGZ → 轴状位 2D 切片 PNG（如果 DenseNet-121 notebook 已经生成过，这里会直接复用跳过）──
def get_axial_slice(vol: np.ndarray, idx: int) -> np.ndarray:
    return vol[:, :, idx]

manifest_rows = []
for sid, mgz_path in tqdm(mgz_map.items(), desc='MGZ -> axial PNG'):
    info = subject_fold[sid]
    cls_name = info['label']
    subj_dir = f'{SLICES_DIR}/{sid}'
    os.makedirs(subj_dir, exist_ok=True)

    existing = sorted(Path(subj_dir).glob('z*.png'))
    if len(existing) >= N_SLICES:
        for p in existing[:N_SLICES]:
            manifest_rows.append({'subject_id': sid, 'path': str(p), 'label': cls_name,
                                   'fold': info['oof_fold']})
        continue

    vol = nib.load(mgz_path).get_fdata()
    n = vol.shape[AXIAL_DIM]
    lo, hi = AXIAL_RANGE
    indices = np.linspace(int(n * lo), int(n * hi) - 1, N_SLICES, dtype=int)
    for i, idx in enumerate(indices):
        out_path = f'{subj_dir}/z{i:02d}.png'
        if not os.path.exists(out_path):
            s = get_axial_slice(vol, idx)
            if s.max() > 1e-3:
                s = (s - s.min()) / (s.max() - s.min() + 1e-8)
                s = (s * 255).astype(np.uint8)
                Image.fromarray(s).save(out_path)
        if os.path.exists(out_path):
            manifest_rows.append({'subject_id': sid, 'path': out_path, 'label': cls_name,
                                   'fold': info['oof_fold']})

manifest_df = pd.DataFrame(manifest_rows)
print(f'\n共生成/找到 {len(manifest_df)} 张切片，覆盖 {manifest_df["subject_id"].nunique()} 个受试者')
print(manifest_df.groupby('fold')['subject_id'].nunique())

In [ ]:
# ── Dataset & transforms（DeiT 的 patch16_224 需要 224x224 输入，和其它基线一致）──

def build_transforms(is_train: bool) -> transforms.Compose:
    norm = transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    if is_train:
        return transforms.Compose([
            transforms.Resize(256),
            transforms.RandomCrop(224),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=10),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.0),
            transforms.ToTensor(),
            norm,
        ])
    return transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), norm])


class SliceDataset(Dataset):
    def __init__(self, rows: pd.DataFrame, is_train: bool):
        self.rows = rows.reset_index(drop=True)
        self.transform = build_transforms(is_train)
        self.class_to_idx = {c: i for i, c in enumerate(CLASS_ORDER)}

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows.iloc[idx]
        img = Image.open(row['path']).convert('RGB')
        label = self.class_to_idx[row['label']]
        return self.transform(img), label, row['subject_id']


def collate_fn(batch):
    imgs = torch.stack([b[0] for b in batch])
    labels = torch.tensor([b[1] for b in batch])
    sids = [b[2] for b in batch]
    return imgs, labels, sids


print('Dataset 定义完成。')

In [ ]:
# ── 模型：DeiT-Small（timm），二分类 ──────────────────────────────
# timm 创建模型时直接传 num_classes，会自动建好对应大小的分类头，不需要像
# torchvision 那样手动替换最后一层。冻结策略按参数名里有没有 'head' 判断
# （patch_embed/blocks/norm 都算 backbone）。

def build_model() -> nn.Module:
    return timm.create_model(DEIT_MODEL_NAME, pretrained=True, num_classes=2)


def freeze_features(model: nn.Module) -> None:
    for name, p in model.named_parameters():
        if 'head' not in name:
            p.requires_grad = False


def unfreeze_all(model: nn.Module) -> None:
    for p in model.parameters():
        p.requires_grad = True


class EarlyStopping:
    def __init__(self, patience: int, min_delta: float, path: str):
        self.patience, self.min_delta, self.path = patience, min_delta, path
        self.best_acc, self.counter = 0.0, 0

    def step(self, val_acc: float, model: nn.Module) -> bool:
        if val_acc > self.best_acc + self.min_delta:
            self.best_acc, self.counter = val_acc, 0
            torch.save(model.state_dict(), self.path)
            return False
        self.counter += 1
        return self.counter >= self.patience


print('模型定义完成。')

In [ ]:
# ── 训练 / 验证 / 评估函数 ────────────────────────────────────────

def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels, _ in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type='cuda', enabled=USE_AMP):
            outputs = model(imgs)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
        total_loss += loss.item() * labels.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels, _ in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with autocast(device_type='cuda', enabled=USE_AMP):
            outputs = model(imgs)
            loss = criterion(outputs, labels)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
        total_loss += loss.item() * labels.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def collect_predictions(model, loader):
    model.eval()
    all_labels, all_probs, all_sids = [], [], []
    for imgs, labels, sids in loader:
        imgs = imgs.to(DEVICE)
        with autocast(device_type='cuda', enabled=USE_AMP):
            outputs = model(imgs)
        probs = torch.softmax(outputs, 1).cpu().numpy()
        all_labels.extend(labels.numpy())
        all_probs.append(probs)
        all_sids.extend(sids)
    return np.array(all_labels), np.vstack(all_probs), all_sids


def subject_level_aggregate(labels_arr, probs_arr, sids):
    sid_probs, sid_labels = defaultdict(list), {}
    for i, sid in enumerate(sids):
        sid_probs[sid].append(probs_arr[i])
        sid_labels[sid] = labels_arr[i]
    subj_ids = list(sid_probs.keys())
    subj_probs = np.array([np.mean(sid_probs[s], axis=0) for s in subj_ids])
    subj_labels = np.array([sid_labels[s] for s in subj_ids])
    return subj_labels, subj_probs, subj_ids


def plot_training_curves(history: list, fold: int, best_epoch: int, save_path: str) -> None:
    epochs = [h['epoch_global'] for h in history]
    p1_len = sum(1 for h in history if h['phase'] == 1)
    has_p2 = any(h['phase'] == 2 for h in history)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(f'Training Curves — DeiT-Small baseline (fold {fold})', fontsize=12, fontweight='bold')

    for ax, key_tr, key_vl, ylabel, title in [
        (ax1, 'tr_acc', 'vl_acc', 'Accuracy', 'Accuracy'),
        (ax2, 'tr_loss', 'vl_loss', 'Loss', 'Loss'),
    ]:
        ax.plot(epochs, [h[key_tr] for h in history], 'b-o', markersize=3, label='Train')
        ax.plot(epochs, [h[key_vl] for h in history], 'r-o', markersize=3, label='Val')
        ax.axvspan(0.5, p1_len + 0.5, alpha=0.07, color='blue', label='Phase 1 (frozen)')
        if has_p2:
            ax.axvspan(p1_len + 0.5, max(epochs) + 0.5, alpha=0.07, color='orange', label='Phase 2 (finetune)')
        ax.axvline(x=best_epoch, color='green', linestyle='--', linewidth=1.5, label=f'Best ckpt (E{best_epoch})')
        ax.set_xlabel('Global Epoch'); ax.set_ylabel(ylabel); ax.set_title(title)
        ax.legend(fontsize=8); ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'  训练曲线: {save_path}')


print('训练/评估函数定义完成。')

In [ ]:
# ── 主循环：5 折，每折在其余 4 折上训练，在留出折上评估 ─────────────────

all_folds = sorted(manifest_df['fold'].unique())
oof_rows = []
fold_val_acc = {}

for test_fold in all_folds:
    print(f'\n{"="*60}\n[Fold {test_fold}] 测试折\n{"="*60}')

    test_subj = manifest_df[manifest_df['fold'] == test_fold]['subject_id'].unique()
    trainval_df = manifest_df[manifest_df['fold'] != test_fold]
    trainval_subj = trainval_df['subject_id'].unique()

    def is_val_subject(sid: str) -> bool:
        h = int(hashlib.md5(f'{sid}_valsplit'.encode()).hexdigest(), 16) % 100
        return h < 15

    val_subj = [s for s in trainval_subj if is_val_subject(s)]
    train_subj = [s for s in trainval_subj if s not in set(val_subj)]

    train_rows = trainval_df[trainval_df['subject_id'].isin(train_subj)]
    val_rows = trainval_df[trainval_df['subject_id'].isin(val_subj)]
    test_rows = manifest_df[manifest_df['subject_id'].isin(test_subj)]
    print(f'  train受试者={len(train_subj)}  val受试者={len(val_subj)}  test受试者={len(test_subj)}')

    ldr_kw = dict(num_workers=NUM_WORKERS, persistent_workers=True, pin_memory=True, collate_fn=collate_fn)
    train_loader = DataLoader(SliceDataset(train_rows, True), batch_size=BATCH_SIZE, shuffle=True, **ldr_kw)
    val_loader = DataLoader(SliceDataset(val_rows, False), batch_size=BATCH_SIZE * 2, shuffle=False, **ldr_kw)
    test_loader = DataLoader(SliceDataset(test_rows, False), batch_size=BATCH_SIZE * 2, shuffle=False, **ldr_kw)

    train_labels = train_rows['label'].map({'Nondemented': 0, 'Demented': 1})
    counts = train_labels.value_counts()
    class_weights = torch.tensor(
        [len(train_labels) / (2 * counts.get(0, 1)), len(train_labels) / (2 * counts.get(1, 1))],
        dtype=torch.float32, device=DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    model = build_model().to(DEVICE)
    ckpt_path = f'{CHECKPOINT_DIR}/baseline_deit_fold{test_fold}.pth'
    stopper = EarlyStopping(patience=PATIENCE, min_delta=MIN_DELTA, path=ckpt_path)
    history = []

    # Phase 1: 冻结 backbone，只训 head
    freeze_features(model)
    opt1 = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                              lr=PHASE1_LR, weight_decay=WEIGHT_DECAY)
    sch1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=PHASE1_EPOCHS, eta_min=1e-6)
    scaler = GradScaler('cuda', enabled=USE_AMP)
    for epoch in range(1, PHASE1_EPOCHS + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, opt1, scaler)
        vl_loss, vl_acc = validate(model, val_loader, criterion)
        sch1.step()
        print(f'  [P1 E{epoch:02d}] train_acc={tr_acc:.4f} val_acc={vl_acc:.4f}')
        history.append({'phase': 1, 'epoch': epoch, 'epoch_global': epoch,
                         'tr_acc': tr_acc, 'vl_acc': vl_acc, 'tr_loss': tr_loss, 'vl_loss': vl_loss})
        if stopper.step(vl_acc, model):
            print('  Phase 1 早停。')
            break

    # Phase 2: 全层微调
    model.load_state_dict(torch.load(ckpt_path))
    unfreeze_all(model)
    stopper.counter = 0
    p1_len = len(history)
    backbone_params = [p for n, p in model.named_parameters() if 'head' not in n]
    head_params = [p for n, p in model.named_parameters() if 'head' in n]
    opt2 = torch.optim.AdamW([
        {'params': backbone_params, 'lr': PHASE2_LR_CNN},
        {'params': head_params, 'lr': PHASE2_LR_HEAD},
    ], weight_decay=WEIGHT_DECAY)
    sch2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=PHASE2_EPOCHS, eta_min=1e-8)
    scaler = GradScaler('cuda', enabled=USE_AMP)
    for epoch in range(1, PHASE2_EPOCHS + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, opt2, scaler)
        vl_loss, vl_acc = validate(model, val_loader, criterion)
        sch2.step()
        print(f'  [P2 E{epoch:02d}] train_acc={tr_acc:.4f} val_acc={vl_acc:.4f}')
        history.append({'phase': 2, 'epoch': epoch, 'epoch_global': p1_len + epoch,
                         'tr_acc': tr_acc, 'vl_acc': vl_acc, 'tr_loss': tr_loss, 'vl_loss': vl_loss})
        if stopper.step(vl_acc, model):
            print('  Phase 2 早停。')
            break

    model.load_state_dict(torch.load(ckpt_path))
    fold_val_acc[test_fold] = stopper.best_acc
    print(f'  [Fold {test_fold}] 训练完成，best val_acc={stopper.best_acc:.4f}')

    best_epoch = next((h['epoch_global'] for h in history if abs(h['vl_acc'] - stopper.best_acc) < 1e-6),
                       history[-1]['epoch_global'])
    curve_path = f'{RESULTS_DIR}/training_curve_baseline_deit_fold{test_fold}.png'
    plot_training_curves(history, test_fold, best_epoch, curve_path)
    display(IPImage(curve_path))

    labels_arr, probs_arr, sids = collect_predictions(model, test_loader)
    subj_labels, subj_probs, subj_ids = subject_level_aggregate(labels_arr, probs_arr, sids)
    for sid, lbl, probs in zip(subj_ids, subj_labels, subj_probs):
        oof_rows.append({'subject_id': sid, 'label': CLASS_ORDER[lbl],
                          'prob_demented': probs[1], 'fold': test_fold})

oof_df = pd.DataFrame(oof_rows)
oof_csv = f'{RESULTS_DIR}/oof_predictions_baseline_deit.csv'
oof_df.to_csv(oof_csv, index=False)
print(f'\n保存基线 out-of-fold 预测: {oof_csv} ({len(oof_df)} 例)')

In [ ]:
# ── 评估：和其余基线/ANFIS 用完全相同的指标定义，方便直接放一张表里比较 ──────

labels_binary = (oof_df['label'] == 'Demented').astype(int).to_numpy()
probs_demented = oof_df['prob_demented'].to_numpy()
preds = (probs_demented >= 0.5).astype(int)

auc = roc_auc_score(labels_binary, probs_demented)
cm = confusion_matrix(labels_binary, preds)
tn, fp, fn, tp = cm.ravel()
accuracy = (tp + tn) / (tp + tn + fp + fn)
sensitivity = tp / (tp + fn + 1e-9)
specificity = tn / (tn + fp + 1e-9)

print(f'\n=== DeiT-Small 基线 (out-of-fold, N={len(oof_df)}) ===')
print(f'Accuracy    : {accuracy:.4f}')
print(f'AUC-ROC     : {auc:.4f}')
print(f'Sensitivity : {sensitivity:.4f}')
print(f'Specificity : {specificity:.4f}')
print(classification_report(labels_binary, preds, target_names=CLASS_ORDER, digits=4))

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im, ax=ax)
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(CLASS_ORDER, rotation=20, ha='right'); ax.set_yticklabels(CLASS_ORDER)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix — DeiT-Small baseline (OASIS-1)')
th = cm.max() / 2
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > th else 'black', fontsize=13)
plt.tight_layout()
cm_path = f'{RESULTS_DIR}/confusion_matrix_baseline_deit.png'
plt.savefig(cm_path, dpi=150)
plt.close()
display(IPImage(cm_path))
print(f'混淆矩阵图: {cm_path}')

print('\n各折验证集 accuracy（训练过程中早停时刻的最佳值，供参考）:')
for f, acc in sorted(fold_val_acc.items()):
    print(f'  fold {f}: {acc:.4f}')